## Merge AIIE → crosswalk → CPS

# AIIE

We create and validate a new measure of an occupation's exposure to AI that we call the AI Occupational Exposure (AIOE). We use the AIOE to construct a measure of AI exposure at the industry level, which we call the AI Industry Exposure (AIIE) and a measure of AI exposure at the county level, which we call the AI Geographic Exposure (AIGE). We also describe several ways in which the AIOE can be used to create firm level measures of AI exposure. We validate the measures and describe how they can be used in different applications by management, organization and strategy scholars.

Source: https://sms.onlinelibrary.wiley.com/doi/full/10.1002/smj.3286 https://github.com/AIOE-Data https://www.census.gov/topics/employment/industry-occupation/guidance/code-lists.html

## Pipeline:

1. Load industry-level AI exposure (`AIIE`, Appendix B of the AIOE appendix) — keyed by 4-digit NAICS.
2. Clean the 2022 Census→NAICS crosswalk into (Census `IND`, NAICS-pattern) pairs.
3. Build a 2017→2022 `IND` remap from the `2022 Ind Code Changes` sheet so pre-2023 CPS rows can match.
4. For each Census `IND`, find its AIIE — narrow match (AIIE NAICS inside the IND's NAICS scope) with a broader-parent fallback.
5. Left-join onto CPS on the 2022-normalized `IND`.


In [1]:
import re
import pandas as pd

aiie = pd.read_excel("data/AIOE_DataAppendix.xlsx", sheet_name="Appendix B")
aiie.head()

,NAICS,Industry Title,AIIE
0,1133,Logging,-1.360161
1,1151,Support Activities for Crop Production,-2.165304
2,1152,Support Activities for Animal Production,-1.149307
3,2111,Oil and Gas Extraction,0.668615
4,2121,Coal Mining,-1.146242


In [2]:
crosswalk_clean = (
    pd.read_excel(
        "data/2022-Census-Industry-Code-List-with-Crosswalk.xlsx",
        sheet_name="2022 Census Ind Code List ",
    )
    .dropna(how="all")
    .rename(
        columns={
            "Industry 2022 Description": "industry_desc",
            "2022 Census Industry Code": "census_ind_raw",
            "2022 NAICS Code": "naics_raw",
        }
    )
)

is_leaf = ~crosswalk_clean["census_ind_raw"].astype(str).str.contains("-", na=False)
crosswalk_clean = crosswalk_clean.loc[is_leaf].copy()
crosswalk_clean["IND"] = pd.to_numeric(
    crosswalk_clean["census_ind_raw"], errors="coerce"
).astype("Int64")
crosswalk_clean = crosswalk_clean.dropna(subset=["IND"])


def parse_naics_patterns(raw):
    """Return the list of NAICS digit-prefixes encoded in one crosswalk cell."""
    if pd.isna(raw):
        return []
    s = str(raw)
    s = re.sub(
        r"(?i)part of|pt\.?", "", s
    )  # "Part of 311", "pt. 92811" -> "311", "92811"
    s = re.sub(r"(?i)exc\..*", "", s)  # "517 exc. 517111" -> "517"
    patterns = []
    for chunk in s.split(","):
        m = re.match(r"\s*(\d+)", chunk)
        if m:
            patterns.append(m.group(1))
    return patterns


crosswalk_clean["naics_patterns"] = crosswalk_clean["naics_raw"].apply(
    parse_naics_patterns
)
crosswalk_clean[["IND", "industry_desc", "naics_raw", "naics_patterns"]].head(10)

,IND,industry_desc,naics_raw,naics_patterns
4,170,Crop production,111,[111]
5,180,Animal production and aquaculture,112,[112]
6,190,Forestry except logging,"1131, 1132","[1131, 1132]"
7,270,Logging,1133,[1133]
8,280,"Fishing, hunting and trapping",114,[114]
9,290,Support activities for agriculture and forestry,115,[115]
13,370,Oil and gas extraction,211,[211]
14,380,Coal mining,2121,[2121]
15,390,Metal ore mining,2122,[2122]
16,470,Nonmetallic mineral mining and quarrying,2123,[2123]


In [3]:
code_changes = (
    pd.read_excel(
        "data/2022-Census-Industry-Code-List-with-Crosswalk.xlsx",
        sheet_name="2022 Ind Code Changes",
        header=2,
    )
    .dropna(subset=["2017 Census Industry Code"])
    .astype({"2017 Census Industry Code": int, "2022 Census Industry Code": int})
)
ind_2017_to_2022 = dict(
    zip(
        code_changes["2017 Census Industry Code"],
        code_changes["2022 Census Industry Code"],
    )
)
print(f"{len(ind_2017_to_2022)} Census IND codes remapped from 2017 to 2022")
code_changes.head()

38 Census IND codes remapped from 2017 to 2022


,2017 Census Industry Code,2017 Census Industry Description,2022 Census Industry Code,2022 Census Industry Description
0,4680,Other motor vehicle dealers,4681,Other motor vehicle dealers
1,4690,"Automotive parts, accessories, and tire stores",4691,"Automotive parts, accessories, and tire retailers"
2,4770,Furniture and home furnishings stores,4771,Furniture and home furnishings retailers
3,4780,Household appliance stores,4796,Electronics and appliance retailers
4,4795,Electronics stores,4796,Electronics and appliance retailers


In [4]:
# AIIE is published at the 4-digit NAICS level. Census INDs map to NAICS at varying
# granularities, so we do a two-pass match:

aiie_lookup = dict(zip(aiie["NAICS"].astype(str), aiie["AIIE"]))
aiie_keys_longest_first = sorted(aiie_lookup, key=len, reverse=True)


def ind_to_aiie(patterns):
    narrow = [
        aiie_lookup[k]
        for p in patterns
        for k in aiie_keys_longest_first
        if k.startswith(p)
    ]
    if narrow:
        return sum(narrow) / len(narrow)
    for p in patterns:
        for k in aiie_keys_longest_first:  # longest prefix wins
            if p.startswith(k):
                return aiie_lookup[k]
    return None


crosswalk_clean["AIIE"] = crosswalk_clean["naics_patterns"].apply(ind_to_aiie)
ind_aiie = (
    crosswalk_clean.dropna(subset=["AIIE"])
    .astype({"IND": "int64"})[["IND", "industry_desc", "AIIE"]]
    .reset_index(drop=True)
)
print(f"Census INDs with AIIE: {len(ind_aiie)} / {len(crosswalk_clean)}")
ind_aiie.head()

Census INDs with AIIE: 182 / 268


,IND,industry_desc,AIIE
0,270,Logging,-1.360161
1,290,Support activities for agriculture and forestry,-1.657306
2,370,Oil and gas extraction,0.668615
3,380,Coal mining,-1.146242
4,390,Metal ore mining,-0.892403


In [5]:
df = pd.read_csv("data/cps_00001.csv")
df["IND_2022"] = df["IND"].map(lambda x: ind_2017_to_2022.get(x, x))
df_aiie = df.merge(
    ind_aiie.rename(columns={"IND": "IND_2022"}),
    on="IND_2022",
    how="left",
)

employed = df_aiie["IND"] != 0  # IND == 0 is NIU (not employed / not in universe)
n_emp = int(employed.sum())
n_cov = int((employed & df_aiie["AIIE"].notna()).sum())
print(f"CPS rows: {len(df_aiie):,}")
print(f"Employed (IND != 0): {n_emp:,}")
print(f"  with AIIE: {n_cov:,} ({n_cov / n_emp:.1%})")

missing_inds = (
    df_aiie.loc[employed & df_aiie["AIIE"].isna(), "IND"].value_counts().head(10)
)
print("\nTop remaining unmatched CPS INDs:")
missing_inds

CPS rows: 1,528,045
Employed (IND != 0): 775,795
  with AIIE: 599,044 (77.2%)

Top remaining unmatched CPS INDs:


IND
8190    16333
9470    12324
4970     6408
170      6276
9480     6234
9370     6206
6990     6129
7070     6087
4971     5560
9590     5217
Name: count, dtype: int64

In [ ]:
df_aiie.to_csv("data/data_with_aiie.csv", index=False)